# PCMCI+ Effectome Sweep

This notebook mirrors the `pcmciplus` simulation template, but runs on effectome window artifacts.
It sweeps `tau_max`, estimates a PCMCI+ matrix for each window, and saves per-depth matrix stacks
plus summary tables under `outputs/notebooks/pcmciplus/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'src' / 'effectome').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate effectome project root')

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from effectome.experiments import collapse_tigramite_results, find_project_root, plot_depth_summary, summarize_stack
from effectome.utils.io import load_artifact, save_artifact, save_matrices
from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.pcmci import PCMCI

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
WINDOWS_PATH = PROJECT_ROOT / 'outputs' / 'artifacts' / 'windows.pkl'
TAU_MIN = 1
TAU_MAX = 7
TAU_RANGE = list(range(TAU_MIN, TAU_MAX + 1))
PC_ALPHA = 0.05
EDGE_ALPHA = 0.05
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebooks' / 'pcmciplus'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Windows path: {WINDOWS_PATH}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
windows = load_artifact(WINDOWS_PATH)
print(f'Loaded {windows.n_windows} windows with shape {windows.segments.shape}')


def estimate_pcmci_stack(windows, tau_max: int) -> np.ndarray:
    matrices = []
    for segment in tqdm(windows.segments, desc=f'PCMCI+ tau={tau_max}'):
        dataframe = pp.DataFrame(np.asarray(segment, dtype=np.float64))
        pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=0)
        results = pcmci.run_pcmciplus(tau_min=TAU_MIN, tau_max=tau_max, pc_alpha=PC_ALPHA)
        matrices.append(collapse_tigramite_results(results, alpha=EDGE_ALPHA, tau_min=TAU_MIN))
    return np.asarray(matrices, dtype=np.float32)


In [ ]:
results = {}
summary_rows = []

for tau in TAU_RANGE:
    matrices = estimate_pcmci_stack(windows, tau)
    save_matrices(matrices, OUTPUT_DIR / f'pcmciplus_tau_{tau}.npz')
    results[tau] = matrices
    stats = summarize_stack(matrices)
    stats['tau'] = tau
    summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows).sort_values('tau').reset_index(drop=True)
summary_df.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
summary_df


In [ ]:
fig = plot_depth_summary(summary_df, 'tau', 'PCMCI+ Effectome Summary vs tau_max')
fig.savefig(OUTPUT_DIR / 'pcmciplus_tau_summary.png', dpi=150, bbox_inches='tight')
plt.show()

save_artifact(
    {'method': 'pcmciplus', 'taus': TAU_RANGE, 'summary': summary_df.to_dict(orient='records')},
    OUTPUT_DIR / 'summary.pkl',
)
print('Saved notebook outputs to', OUTPUT_DIR)
